In [1]:
!pip install rasterio contextily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 60.2 MB/s eta 0:00:00


In [5]:
import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import ee
import geemap
import json
from datetime import datetime
import requests
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import zipfile
import rasterio
from PIL import Image
import io
from google.colab import files

# Cập nhật danh sách tỉnh ĐBSCL
MEKONG_PROVINCES = [
    'An Giang', 'Ben Tre', 'Ca Mau', 'Can Tho city', 'Dong Thap',
    'Hau Giang', 'Kien Giang', 'Long An', 'Soc Trang',
    'Tien Giang', 'Tra Vinh', 'Vinh Long', 'Bac Lieu'
]

# Xác thực và khởi tạo Earth Engine
try:
    ee.Initialize(project='ee-python-api-471906')
    print("Earth Engine initialized successfully")
except Exception as e:
    print(f"Failed to initialize Earth Engine: {e}")
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

def get_mekong_region():
    """Lấy geometry của vùng ĐBSCL"""
    try:
        provinces = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam'))
        mekong_fc = provinces.filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))
        return mekong_fc.union().geometry()
    except Exception as e:
        print(f"Error getting Mekong region: {e}")
        raise

def get_s2_collection(region, start_date, end_date, cloud_filter=30):
    try:
        collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                      .filterBounds(region)
                      .filterDate(start_date, end_date)
                      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_filter))
                      .sort('CLOUDY_PIXEL_PERCENTAGE'))
        size = collection.size().getInfo()
        if size == 0:
            print(f"No images found for {start_date} to {end_date}")
            return None
        print(f"Found {size} images")
        return collection
    except Exception as e:
        print(f"Error getting Sentinel-2 collection: {e}")
        return None

def download_image_directly(image, output_subdir, scale=100, bands=None):
    """Tải ảnh trực tiếp về local qua EE API"""
    try:
        if bands is None:
            bands = ['B4', 'B3', 'B2']  # RGB bands

        image_id = image.get('system:index').getInfo()

        url = image.select(bands).getDownloadURL({
            'region': image.geometry(), # Sử dụng geometry của ảnh đã được clip
            'scale': scale,
            'format': 'GEO_TIFF',
            'crs': 'EPSG:4326',
            'filePerBand': False
        })

        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            content_type = response.headers.get('content-type', '')

            if 'zip' in content_type:
                filename = f"{image_id.split('/')[-1]}.zip"
                filepath = os.path.join(output_subdir, filename)

                with open(filepath, 'wb') as f:
                    f.write(response.content)

                with zipfile.ZipFile(filepath, 'r') as zip_ref:
                    zip_ref.extractall(output_subdir)

                os.remove(filepath)

                return filepath.replace('.zip', '.tif')

            else:
                filename = f"{image_id.split('/')[-1]}.tif"
                filepath = os.path.join(output_subdir, filename)

                with open(filepath, 'wb') as f:
                    f.write(response.content)

                return filepath

        else:
            return None

    except Exception as e:
        return None

def download_batch_images(collection, region, output_dir, max_images=None, scale=100):
    """Tải hàng loạt ảnh với multi-threading"""

    if max_images == None:
        images_list = collection.toList(collection.size())
        total_images = images_list.size().getInfo()
    else:
        images_list = collection.toList(max_images)
        total_images = min(max_images, images_list.size().getInfo())
    results = []

    with tqdm(total=total_images, desc="Tải ảnh", unit="ảnh") as pbar:
        for i in range(total_images):
            try:
                img = ee.Image(images_list.get(i)).clip(region)

                # Tạo thư mục con ngay tại đây
                image_date_millis = img.get('system:time_start').getInfo()
                date_time_str = datetime.fromtimestamp(image_date_millis / 1000).strftime('%Y-%m-%d_%H-%M-%S')
                image_subdir = os.path.join(output_dir, f"S2_{date_time_str}")
                os.makedirs(image_subdir, exist_ok=True)

                # Cập nhật thông tin trong thanh tiến trình
                pbar.set_postfix_str(f"Processing: S2_{date_time_str}")

                # Tải ảnh, lưu metadata và quicklook
                result = download_image_directly(img, image_subdir, scale)
                save_image_metadata(img, image_subdir)
                create_matplotlib_quicklook(img, region, image_subdir)

                if result:
                    results.append(result)
                time.sleep(2)  # Chờ giữa các lần tải
                pbar.update(1)
            except Exception as e:
                pbar.write(f"❌ Lỗi xử lý ảnh {i+1}: {e}")

    return results

def save_image_metadata(image, output_subdir):
    """Lưu metadata của ảnh"""
    try:
        image_id = image.get('system:index').getInfo()
        properties = image.toDictionary().getInfo()

        metadata_filename = f"{image_id.split('/')[-1]}_metadata.json"
        metadata_path = os.path.join(output_subdir, metadata_filename)

        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(properties, f, ensure_ascii=False, indent=4)

        return metadata_path

    except Exception as e:
        return None

def create_matplotlib_quicklook(image, region, output_subdir, scale=200):
    """
    Tạo ảnh quicklook RGB bằng Matplotlib, bao gồm đường viền ĐBSCL
    và lưu vào thư mục con của ảnh.
    """
    try:
        image_id = image.get('system:index').getInfo()
        image_date_millis = image.get('system:time_start').getInfo()

        rgb_img_vis = image.select(['B4', 'B3', 'B2']).visualize(
            min=0,
            max=3000,
            gamma=1.4
        )

        rgb_array = geemap.ee_to_numpy(
            rgb_img_vis,
            region=region,
            scale=scale
        )
        rgb_array = np.nan_to_num(rgb_array, nan=255)
        rgb_array[rgb_array == 0] = 255

        image_date_title = datetime.fromtimestamp(image_date_millis / 1000).strftime('%d-%m-%Y %H:%M:%S')

        bounds = region.bounds().getInfo()["coordinates"][0]
        minx = min([c[0] for c in bounds])
        maxx = max([c[0] for c in bounds])
        miny = min([c[1] for c in bounds])
        maxy = max([c[1] for c in bounds])
        if miny > maxy:
            miny, maxy = maxy, miny

        fig, ax = plt.subplots(figsize=(12, 12))
        ax.set_aspect('equal')
        ax.imshow(rgb_array, extent=[minx, maxx, miny, maxy])


        mekong_boundary_coords = region.getInfo()['coordinates']
        for polygon in mekong_boundary_coords:
            for ring in polygon:
                lon = [point[0] for point in ring]
                lat = [point[1] for point in ring]
                ax.plot(lon, lat, color='black', linewidth=1.5, alpha=0.8)

        ax.set_title(f'Ảnh vệ tinh ĐBSCL ({image_date_title})', fontsize=16)
        ax.set_xlabel('Kinh độ', fontsize=12)
        ax.set_ylabel('Vĩ độ', fontsize=12)
        ax.autoscale(enable=True, axis='both', tight=True)
        plt.grid(True, linestyle='--', alpha=0.6)

        filename = f"{image_id.split('/')[-1]}_preview.png"
        filepath = os.path.join(output_subdir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close(fig)

        return filepath

    except Exception as e:
        return None
import os
from google.colab import files

def main():
    year = 2022
    months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
    mekong_region = get_mekong_region()

    # Vòng lặp tải và nén từng tháng, NHƯNG KHÔNG TẢI VỀ
    for idx in range(len(months)):
        # ... (Các dòng code tải và nén từng tháng) ...
        OUTPUT_DIR = f"/content/dbscl-sentinel-2_{year}-{months[idx]}"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        start_date = f'{year}-{months[idx]}-01'
        if idx == len(months) - 1:
            end_date = f'{year + 1}-01-01'
        else:
            end_date = f'{year}-{months[idx+1]}-01'
        collection = get_s2_collection(mekong_region, start_date, end_date, cloud_filter=50)

        if collection.size().getInfo() == 0:
            print(f"Không tìm thấy ảnh cho tháng {months[idx]}. Bỏ qua...")
            continue

        print(f"🚀 Bắt đầu tải ảnh vệ tinh Sentinel-2 cho tháng {months[idx]}/{year}...")
        downloaded_files = download_batch_images(
            collection, mekong_region, OUTPUT_DIR, scale=200
        )
        print(f"✅ Đã hoàn tất xử lý {len(downloaded_files)} ảnh trong tháng.")

        # CHỈ NÉN file của tháng hiện tại.
        # Dòng code này sẽ tạo ra nhiều file zip
        # zip_path = OUTPUT_DIR + '.zip'
        # os.system(f'zip -r {zip_path} {OUTPUT_DIR}')

    # --- Kết thúc vòng lặp ---

    # Nén TẤT CẢ các folder của các tháng thành một file zip duy nhất
    print("📦 Đang nén tất cả các file zip thành một...")
    all_months_zip = f'/content/sentinel-2_{year}_all_months.zip'
    os.system(f'zip -r {all_months_zip} /content/dbscl-sentinel-2_{year}*')

    # Tải file zip cuối cùng về
    print("⬇️ Bắt đầu tải file cuối cùng...")
    files.download(all_months_zip)

if __name__ == "__main__":
    main()

Earth Engine initialized successfully
Found 87 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-2 cho tháng 01/2022...


Tải ảnh: 100%|██████████| 87/87 [51:39<00:00, 35.62s/ảnh, Processing: S2_2022-01-27_03-25-25]


✅ Đã hoàn tất xử lý 87 ảnh trong tháng.
Found 45 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-2 cho tháng 02/2022...


Tải ảnh: 100%|██████████| 45/45 [28:20<00:00, 37.79s/ảnh, Processing: S2_2022-02-14_03-35-44]


✅ Đã hoàn tất xử lý 45 ảnh trong tháng.
Found 59 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-2 cho tháng 03/2022...


...

✅ Đã hoàn tất xử lý 11 ảnh trong tháng.
Found 45 images
🚀 Bắt đầu tải ảnh vệ tinh Sentinel-2 cho tháng 12/2022...


Tải ảnh: 100%|██████████| 45/45 [28:14<00:00, 37.66s/ảnh, Processing: S2_2022-12-26_03-35-11]


✅ Đã hoàn tất xử lý 44 ảnh trong tháng.
📦 Đang nén tất cả các file zip thành một...
⬇️ Bắt đầu tải file cuối cùng...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>